# KimyGuide: TF-IDF Goal-Based Recommendation Prototype

This notebook implements and evaluates the **core feature prototype** for the KimyGuide project:  
a **goal-driven, content-based recommendation system** designed to provide meaningful learning recommendations from the very first interaction (cold-start scenario).

The prototype focuses on:
- using **textual course metadata** (titles and descriptions),
- representing both learner goals and course content with **TF-IDF vectors**,
- ranking courses using **cosine similarity**, and
- evaluating recommendation quality against a simple baseline.

This notebook supports the **Feature Prototype** and **Evaluation** components of the CM3005 preliminary submission.


## 1. Setup and Environment

This section sets up the notebook environment and defines the project root and data paths.
Explicit path handling is used to ensure reproducibility across development environments,
including local machines and cloud-based notebooks.

All paths are resolved relative to the project root to avoid dependency on the notebook's working directory.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


PROJECT_ROOT = Path.cwd().resolve().parents[0]

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "courses_mooclike.csv"

print("Notebook cwd:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())


Notebook cwd: /Users/mac/kimyguide/notebooks
Project root: /Users/mac/kimyguide
Dataset path: /Users/mac/kimyguide/data/raw/courses_mooclike.csv
Exists: True


## 2. Dataset Loading and Preparation

For the feature prototype, a **synthetic MOOC-like dataset** is used.
The dataset mimics real online course catalog data by including:

- course titles,
- rich textual descriptions,
- topic tags,
- difficulty level,
- estimated duration,
- delivery format.

This approach allows rapid prototyping and controlled evaluation while preserving
compatibility with the planned final dataset (MOOCCube), which will be used in the full project.

A combined `text` field is created by concatenating the course title and description,
which serves as the input for TF-IDF vectorization.


In [ ]:
df = pd.read_csv(DATA_PATH)

expected = {"course_id", "title", "description", "tags", "level", "duration_hours", "format"}
missing = expected - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}. Found: {df.columns.tolist()}")

df["text"] = (df["title"].fillna("") + " " + df["description"].fillna("")).str.strip()

df.head(3)


,course_id,title,description,tags,level,duration_hours,format,text
0,1,Deep Learning: From Basics to Applications,Learn backpropagation and neural networks whil...,"Deep Learning, neural networks, regularization",Beginner,7,Project,Deep Learning: From Basics to Applications Lea...
1,2,Deep Learning: From Basics to Applications,This course focuses on regularization and tran...,"Deep Learning, RNNs, transformers",Beginner,12,Video,Deep Learning: From Basics to Applications Thi...
2,3,Practical Python for Real Projects,"You will explore APIs, numpy, and pandas throu...","Python, data structures, APIs",Intermediate,20,Video,Practical Python for Real Projects You will ex...


## 3. Dataset Overview

Before building the recommendation model, we perform a brief exploratory check
to confirm that the dataset:

- contains a reasonable number of items,
- spans multiple difficulty levels and formats,
- resembles realistic MOOC catalog data.

This step helps validate that the dataset is suitable for demonstrating the prototype.


In [ ]:
print("Rows:", len(df))
print("Unique titles:", df["title"].nunique())
print("Levels:", df["level"].value_counts().to_dict())
print("Formats:", df["format"].value_counts().to_dict())

# few examples
df[["course_id", "title", "level", "format", "tags"]].sample(5, random_state=42)


Rows: 120
Unique titles: 61
Levels: {'Beginner': 63, 'Intermediate': 39, 'Advanced': 18}
Formats: {'Mixed': 34, 'Video': 30, 'Reading': 30, 'Project': 26}


,course_id,title,level,format,tags
44,45,Applied Data Science and Evaluation,Beginner,Video,"Data Science, hypothesis testing, data cleaning"
47,48,Data Science Foundations,Beginner,Project,"Data Science, pipelines, data cleaning"
4,5,Hands-on Cloud & Deployment Bootcamp,Beginner,Mixed,"Cloud & Deployment, reproducibility, APIs"
55,56,Time Series with Python,Advanced,Mixed,"Time Series, ARIMA, forecasting"
26,27,Building Systems with Data Science,Beginner,Reading,"Data Science, hypothesis testing, pipelines"


## 4. Text Representation Using TF-IDF

To represent both learner goals and course content, we use **Term Frequency–Inverse Document Frequency (TF-IDF)**.

TF-IDF is chosen because:
- it is lightweight and interpretable,
- it performs well for content-based recommendation in cold-start settings,
- it provides a strong baseline before introducing more complex embedding models.

Both unigrams and bigrams are included to capture important phrases
(e.g. *"neural networks"*, *"window functions"*).


In [10]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

X = vectorizer.fit_transform(df["text"])
print("TF-IDF matrix shape:", X.shape)


TF-IDF matrix shape: (120, 844)


## 5. Recommendation Methods

Two recommendation strategies are implemented:

### 5.1 TF-IDF Content-Based Recommender
The main prototype model ranks courses by computing the cosine similarity
between the TF-IDF representation of a learner's goal and each course description.

### 5.2 Random Baseline
A random recommender is used as a simple baseline.
This provides a lower bound for performance and helps contextualize the effectiveness
of the TF-IDF approach.

Comparing against a baseline is essential for demonstrating that the prototype
adds value beyond chance.


In [11]:
def recommend_tfidf(goal: str, top_k: int = 5) -> pd.DataFrame:
    g = vectorizer.transform([goal])
    sims = cosine_similarity(g, X).ravel()
    idx = np.argsort(-sims)[:top_k]
    out = df.iloc[idx].copy()
    out["score"] = sims[idx]
    out["goal"] = goal
    return out[["goal", "course_id", "title", "score", "tags", "level", "format"]]

def recommend_random(goal: str, top_k: int = 5, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(df), size=top_k, replace=False)
    out = df.iloc[idx].copy()
    out["score"] = np.nan
    out["goal"] = goal
    return out[["goal", "course_id", "title", "score", "tags", "level", "format"]]


## 6. Example Recommendation Output

This section demonstrates a single recommendation query to illustrate:

- the end-to-end recommendation pipeline,
- the structure of the output,
- how ranked results align with the learner's stated goal.

This example is also representative of what is shown in the prototype demo video.


In [12]:
goal = "I want to learn SQL joins and window functions"
recommend_tfidf(goal, top_k=5)


,goal,course_id,title,score,tags,level,format
78,I want to learn SQL joins and window functions,79,Introduction to SQL,0.323403,"SQL, SELECT queries, joins",Beginner,Mixed
3,I want to learn SQL joins and window functions,4,SQL with Python,0.312618,"SQL, data modeling, indexes",Beginner,Mixed
96,I want to learn SQL joins and window functions,97,Practical SQL for Real Projects,0.305238,"SQL, group by, indexes",Intermediate,Reading
115,I want to learn SQL joins and window functions,116,Building Systems with SQL,0.167464,"SQL, SELECT queries, group by",Beginner,Reading
103,I want to learn SQL joins and window functions,104,Applied SQL and Evaluation,0.164964,"SQL, SELECT queries, joins",Intermediate,Project


## 7. Proxy Relevance Definition

Because the dataset is synthetic and does not contain explicit user feedback,
a **proxy relevance definition** is used for evaluation.

A recommendation is considered relevant if:
- the detected topic from the learner goal
- appears in the course's topic tags.

This approach provides a reasonable approximation for preliminary evaluation
and allows comparison between different recommendation strategies.

The final project will replace this proxy with:
- real interaction data,
- cold-start splits,
- ranking metrics such as Recall@k and NDCG@k.


In [ ]:
TOPICS = [
    "Machine Learning",
    "Deep Learning",
    "Natural Language Processing",
    "Data Science",
    "Python",
    "SQL",
    "Recommender Systems",
    "Product Analytics",
    "Cloud & Deployment",
    "Time Series",
]

def detect_topic(goal: str) -> str | None:
    g = goal.lower()
    # simple keyword rules
    rules = [
        ("recommender", "Recommender Systems"),
        ("recommendation", "Recommender Systems"),
        ("deep learning", "Deep Learning"),
        ("neural", "Deep Learning"),
        ("nlp", "Natural Language Processing"),
        ("language", "Natural Language Processing"),
        ("sql", "SQL"),
        ("python", "Python"),
        ("time series", "Time Series"),
        ("forecast", "Time Series"),
        ("cloud", "Cloud & Deployment"),
        ("docker", "Cloud & Deployment"),
        ("analytics", "Product Analytics"),
        ("ab test", "Product Analytics"),
        ("a/b", "Product Analytics"),
        ("machine learning", "Machine Learning"),
        ("regression", "Machine Learning"),
        ("classification", "Machine Learning"),
        ("data science", "Data Science"),
        ("eda", "Data Science"),
        ("statistics", "Data Science"),
    ]
    for key, topic in rules:
        if key in g:
            return topic
    return None

def relevant_course_mask(topic: str) -> pd.Series:
    # tag match (case-insensitive)
    return df["tags"].str.lower().str.contains(topic.lower())


## 8. Evaluation Metrics

Two standard information retrieval metrics are used:

- **Precision@k**: the proportion of recommended items in the top-k that are relevant.
- **Recall@k**: the proportion of all relevant items that appear in the top-k recommendations.

These metrics are appropriate for ranking-based recommendation tasks and
are widely used in educational recommender system literature.


In [14]:
def precision_recall_at_k(recs: pd.DataFrame, topic: str, k: int = 5) -> tuple[float, float]:
    rel_mask = relevant_course_mask(topic)
    relevant_ids = set(df.loc[rel_mask, "course_id"].tolist())

    recommended_ids = recs["course_id"].head(k).tolist()
    hits = sum(1 for cid in recommended_ids if cid in relevant_ids)

    precision = hits / k
    recall = hits / max(len(relevant_ids), 1)
    return precision, recall


## 9. Evaluation Results

The TF-IDF recommender is evaluated across multiple learner goals
and compared against the random baseline.

Results are reported per goal and as an average across goals.
This allows us to assess both consistency and overall effectiveness.


In [15]:
GOALS = [
    "I want to learn SQL joins and window functions",
    "I want to learn deep learning and neural networks",
    "I want to learn recommender systems and cold start",
    "I want to learn NLP embeddings and transformers",
    "I want to learn time series forecasting and anomaly detection",
]

rows = []
k = 5

for goal in GOALS:
    topic = detect_topic(goal)
    if topic is None:
        print("⚠️ Could not detect topic for:", goal)
        continue

    recs_tfidf = recommend_tfidf(goal, top_k=k)
    recs_rand = recommend_random(goal, top_k=k, seed=42)

    p_t, r_t = precision_recall_at_k(recs_tfidf, topic, k=k)
    p_r, r_r = precision_recall_at_k(recs_rand, topic, k=k)

    rows.append({
        "goal": goal,
        "topic_proxy": topic,
        "precision@5_tfidf": round(p_t, 3),
        "recall@5_tfidf": round(r_t, 3),
        "precision@5_random": round(p_r, 3),
        "recall@5_random": round(r_r, 3),
    })

results = pd.DataFrame(rows)
results


,goal,topic_proxy,precision@5_tfidf,recall@5_tfidf,precision@5_random,recall@5_random
0,I want to learn SQL joins and window functions,SQL,1.0,0.625,0.0,0.000
1,I want to learn deep learning and neural networks,Deep Learning,1.0,0.263,0.4,0.105
2,I want to learn recommender systems and cold s...,Recommender Systems,1.0,0.833,0.0,0.000
3,I want to learn NLP embeddings and transformers,Natural Language Processing,1.0,0.417,0.6,0.250
4,I want to learn time series forecasting and an...,Time Series,1.0,0.333,0.0,0.000


## 10. Summary of Results

The averaged evaluation results provide a high-level comparison between
the TF-IDF recommender and the random baseline.

These results are used in the written report to:
- justify the choice of TF-IDF for the prototype,
- highlight strengths and limitations,
- motivate future improvements.


In [16]:
results.mean(numeric_only=True).to_frame("average").T


,precision@5_tfidf,recall@5_tfidf,precision@5_random,recall@5_random
average,1.0,0.4942,0.2,0.071


## 11. Saving Evaluation Outputs

Evaluation results are saved to disk to ensure reproducibility and
to support inclusion in the written report.

Saving intermediate outputs is considered good experimental practice
and facilitates transparent reporting.


In [17]:
out_dir = PROJECT_ROOT / "reports" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(out_dir / "prelim_eval_results.csv", index=False)
print(" Saved:", out_dir / "prelim_eval_results.csv")


 Saved: /Users/mac/kimyguide/reports/figures/prelim_eval_results.csv
